Generate single-model responses for the configured benchmark and save responses with token usage.


In [ ]:
from pathlib import Path
import os
_REPOSITORY_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(_REPOSITORY_ROOT).startswith('\\\\?\\')):
    _REPOSITORY_ROOT = Path('\\\\?\\' + str(_REPOSITORY_ROOT))
_NOTEBOOK_DIR = _REPOSITORY_ROOT / 'experiments/clinical_protocols/single_model_baselines/baseline_claude_opus5'
os.chdir(_NOTEBOOK_DIR)


In [ ]:
import ast
import json
import os
import hashlib
import re
import time
from typing import Dict, Any, List
import pandas as pd
import requests
import anthropic
from tqdm.auto import tqdm


In [ ]:
ANTHROPIC_API_KEY_ENV = 'ANTHROPIC_API_KEY'
API_KEY = ''
FOLDER_ID = ''
LLM_API_URL = ''
INPUT_CSV = 'llmfmed_dataset.csv'
OUTPUT_CSV = 'baseline_roles_claude_opus5.csv'
BASELINE_MODEL_NAME = 'claude_opus5_roles'
BASELINE_MODEL_URI = 'claude-opus-5'
ROLE_COLUMN = 'First Role'
CASE_COMPONENT_COLUMNS = ['Vignette', 'Review of Systems', 'Past Medical History', 'Physical Examination', 'Test Results']
EMERGENCY_LABELS = ['Self-care appropriate', 'Requires non-emergent care', 'Requires emergent care']
SUBCLINICAL_LABELS = ['Cardiology', 'Dermatology', 'Emergency', 'Endocrinology', 'Gastroenterology', 'General Medicine', 'General Surgery', 'Infectious Disease', 'Nephrology', 'Neurology', 'Obstetrics and Gynecology', 'Ophthalmology', 'Orthopedics', 'Other departments', 'Otolaryngology (ENT)', 'Pediatrics', 'Psychiatry', 'Pulmonology', 'Urology']
REQUEST_TIMEOUT = 120
BASE_SLEEP_SECONDS = 2
MAX_RETRIES = 6
TEMPERATURE_PROTOCOL = None
MAX_TOKENS_PROTOCOL = 16000
REASONING_EFFORT = 'high'
USE_ADAPTIVE_THINKING = True

def detect_provider(model_uri: str) -> str:
    if model_uri.startswith('claude-'):
        return 'anthropic'
    if 'aliceai-llm' in model_uri:
        return 'yandex_rest'
    return 'openai_compatible'
BASELINE_PROVIDER = detect_provider(BASELINE_MODEL_URI)
df = pd.read_csv(INPUT_CSV, sep=None, engine='python')
df = df.copy().reset_index(drop=True)
missing_case_columns = [col for col in CASE_COMPONENT_COLUMNS if col not in df.columns]
if missing_case_columns:
    raise ValueError(f'Missing case columns in dataset: {missing_case_columns}')
if ROLE_COLUMN not in df.columns:
    raise ValueError(f'Missing role column in dataset: {ROLE_COLUMN}')

def build_case_text(row_data: Dict[str, Any]) -> str:
    parts: List[str] = []
    for column in CASE_COMPONENT_COLUMNS:
        value = row_data.get(column)
        if pd.isna(value):
            continue
        value_str = str(value).strip()
        if not value_str or value_str.lower() == 'nan':
            continue
        parts.append(f'{column}:\n{value_str}')
    return '\n\n'.join(parts)

def parse_role_cell(role_cell: Any) -> Dict[str, str]:
    raw = '' if pd.isna(role_cell) else str(role_cell).strip()
    if not raw:
        raise ValueError('Encountered an empty role cell.')
    parsed = ast.literal_eval('{' + raw + '}')
    role_name = str(parsed['role_name']).strip()
    role_description = ' '.join(str(parsed['role_description']).split())
    return {'role_name': role_name, 'role_description': role_description}
df['case_text'] = df.apply(lambda row: build_case_text(row.to_dict()), axis=1)
print('Rows:', len(df))
print('Model:', BASELINE_MODEL_NAME, '->', BASELINE_MODEL_URI)
print('Provider:', BASELINE_PROVIDER)
print('Role column:', ROLE_COLUMN)
print('Case columns:', CASE_COMPONENT_COLUMNS)


In [ ]:
llm_cache: Dict[str, Dict[str, Any]] = {}
try:
    anthropic_client = anthropic.Anthropic()
except Exception as e:
    anthropic_client = None
    print(f'WARNING: не удалось создать клиент Anthropic: {e}')
    print('        Задайте ANTHROPIC_API_KEY или выполните `ant auth login`.')
print('anthropic SDK:', anthropic.__version__)

def make_cache_key(provider: str, model_uri: str, system_text: str, user_text: str, temperature: Any, max_tokens: int) -> str:
    payload = {'provider': provider, 'model_uri': model_uri, 'system_text': system_text, 'user_text': user_text, 'temperature': temperature, 'max_tokens': max_tokens, 'effort': REASONING_EFFORT, 'thinking': 'adaptive' if USE_ADAPTIVE_THINKING else 'disabled'}
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()

def _build_token_dict(prompt_tokens: int=0, completion_tokens: int=0, total_tokens: int=0) -> Dict[str, int]:
    return {'prompt_tokens': int(prompt_tokens or 0), 'completion_tokens': int(completion_tokens or 0), 'total_tokens': int(total_tokens or 0)}

def _extract_anthropic_text(message: Any) -> str:
    parts: List[str] = []
    for block in getattr(message, 'content', []) or []:
        if getattr(block, 'type', None) == 'text':
            parts.append(getattr(block, 'text', '') or '')
    return '\n'.join((p for p in parts if p)).strip()

def _extract_anthropic_usage(usage: Any) -> Dict[str, int]:
    if usage is None:
        return _build_token_dict()
    input_tokens = int(getattr(usage, 'input_tokens', 0) or 0)
    output_tokens = int(getattr(usage, 'output_tokens', 0) or 0)
    cache_read = int(getattr(usage, 'cache_read_input_tokens', 0) or 0)
    cache_write = int(getattr(usage, 'cache_creation_input_tokens', 0) or 0)
    prompt_tokens = input_tokens + cache_read + cache_write
    return _build_token_dict(prompt_tokens, output_tokens, prompt_tokens + output_tokens)

def call_anthropic_llm(model_uri: str, system_text: str, user_text: str, temperature: Any, max_tokens: int, retries: int=MAX_RETRIES) -> Dict[str, Any]:
    if anthropic_client is None:
        raise RuntimeError('Клиент Anthropic не создан: нет учётных данных. Задайте ANTHROPIC_API_KEY или выполните `ant auth login`.')
    cache_key = make_cache_key('anthropic', model_uri, system_text, user_text, temperature, max_tokens)
    if cache_key in llm_cache:
        cached = llm_cache[cache_key]
        return {'text': cached['text'], 'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'cached': True, 'stop_reason': cached.get('stop_reason', ''), 'truncated': cached.get('truncated', False)}
    kwargs: Dict[str, Any] = {'model': model_uri, 'max_tokens': max_tokens, 'system': system_text, 'messages': [{'role': 'user', 'content': user_text}], 'thinking': {'type': 'adaptive'} if USE_ADAPTIVE_THINKING else {'type': 'disabled'}}
    if REASONING_EFFORT:
        kwargs['output_config'] = {'effort': REASONING_EFFORT}
    last_error_text = ''
    for attempt in range(retries):
        try:
            message = anthropic_client.messages.create(**kwargs)
        except TypeError as e:
            raise RuntimeError(f'SDK anthropic {anthropic.__version__} не поддерживает параметры запроса ({e}). Обновите: pip install -U anthropic') from e
        except anthropic.NotFoundError as e:
            raise RuntimeError(f'Неизвестная модель или эндпойнт: {model_uri} ({e})') from e
        except anthropic.AuthenticationError as e:
            raise RuntimeError(f'Учётные данные Anthropic отклонены: {e}') from e
        except anthropic.PermissionDeniedError as e:
            raise RuntimeError(f'Нет доступа к модели {model_uri}: {e}') from e
        except anthropic.RateLimitError as e:
            last_error_text = f'rate limit: {e}'
            wait = BASE_SLEEP_SECONDS * 2 ** attempt
            try:
                wait = max(wait, int(e.response.headers.get('retry-after', '0')))
            except Exception:
                pass
            time.sleep(wait)
            continue
        except anthropic.APIStatusError as e:
            last_error_text = f'HTTP {e.status_code}: {e}'
            if e.status_code >= 500:
                time.sleep(BASE_SLEEP_SECONDS * 2 ** attempt)
                continue
            raise RuntimeError(last_error_text) from e
        except anthropic.APIConnectionError as e:
            last_error_text = f'сеть: {e}'
            time.sleep(BASE_SLEEP_SECONDS * 2 ** attempt)
            continue
        stop_reason = getattr(message, 'stop_reason', '') or ''
        if stop_reason == 'refusal':
            details = getattr(message, 'stop_details', None)
            cat = getattr(details, 'category', None) if details else None
            print(f'WARNING: отказ модели (category={cat}) — кейс останется без протокола')
        raw_text = _extract_anthropic_text(message)
        usage = _extract_anthropic_usage(getattr(message, 'usage', None))
        truncated = stop_reason == 'max_tokens'
        if truncated:
            print(f'WARNING: ответ обрезан по max_tokens={max_tokens} — поднимите MAX_TOKENS_PROTOCOL')
        llm_cache[cache_key] = {'text': raw_text, 'stop_reason': stop_reason, 'truncated': truncated}
        return {'text': raw_text, **usage, 'cached': False, 'stop_reason': stop_reason, 'truncated': truncated}
    raise RuntimeError(f'Anthropic call failed after {retries} retries. Last error: {last_error_text}')

def call_llm(provider: str, model_uri: str, system_text: str, user_text: str, temperature: Any, max_tokens: int) -> Dict[str, Any]:

    def _dispatch() -> Dict[str, Any]:
        if provider == 'anthropic':
            return call_anthropic_llm(model_uri, system_text, user_text, temperature, max_tokens)
        raise ValueError(f'Unknown provider: {provider}')
    response = _dispatch()
    if str(response['text']).strip():
        return response
    llm_cache.pop(make_cache_key(provider, model_uri, system_text, user_text, temperature, max_tokens), None)
    retry = _dispatch()
    for key in ('prompt_tokens', 'completion_tokens', 'total_tokens'):
        retry[key] += response[key]
    if not str(retry['text']).strip():
        print(f'WARNING: empty response after retry (model_uri={model_uri}, max_tokens={max_tokens})')
    return retry


In [ ]:
def build_stage1_system_prompt(role_name: str, role_description: str) -> str:
    return f'You are acting as {role_name}.\n{role_description}\n\nYou are a medical expert.\nAt this stage, you need to evaluate a specific clinical case and provide a final structured protocol based on the case.\n\nUse only information explicitly present in the case. If a detail is absent, write "Not specified"; if a field is unnecessary, write "None".\nReason carefully from the case. Do not invent symptoms or findings absent from the case.\n'.strip()

def build_stage1_user_prompt(case_text: str, author_id: str) -> str:
    emergency_list = json.dumps(EMERGENCY_LABELS)
    subclinical_list = json.dumps(SUBCLINICAL_LABELS)
    return f'Read the clinical case and write one medical protocol.\n\nClinical case:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\nWrite the protocol in plain text. Same field labels and order:\n\nAge:\nSex:\nComplaint:\nAnalysis:\nRequired diagnostic equipment/tests:\nConclusion:\nEmergency type:\nSubclinical specialization:\nRecommendations:\nNotes:\nProtocol Author: {author_id}\n\nField instructions:\nAge: copy from the case if stated; otherwise write "Not specified".\nSex: copy from the case if stated; otherwise write "Not specified".\nComplaint: briefly summarize the presenting problem from the case.\nAnalysis: explain the clinical reasoning, the strongest supporting facts, and the main relevant differential considerations.\nRequired diagnostic equipment/tests: mention only studies or equipment that are actually needed next; if nothing additional is needed, write "None".\nConclusion: give a concise clinical synthesis on the case.\nEmergency type: based on your analysis of this case, emergency type must equal exactly one of {emergency_list}.\nSubclinical specialization: based on your analysis of this case, subclinical specialization must equal exactly one of {subclinical_list}.\nRecommendations: provide practical, case-grounded next steps or management advice. Explicitly address the additional considerations.\nNotes: include only essential caveats; otherwise write "None".\n'.strip()

def strip_author_section(protocol_text: str) -> str:
    if protocol_text is None:
        return ''
    text = str(protocol_text).strip()
    parts = re.split('(?im)^\\s*Protocol Author\\s*:', text, maxsplit=1)
    return parts[0].rstrip()


In [ ]:
def run_baseline_pipeline(input_df: pd.DataFrame) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    for _, row in tqdm(input_df.iterrows(), total=len(input_df), desc='Baseline pipeline'):
        row_data = row.to_dict()
        case_text = row_data['case_text']
        role_data = parse_role_cell(row_data[ROLE_COLUMN])
        role_name = role_data['role_name']
        role_description = role_data['role_description']
        system_text = build_stage1_system_prompt(role_name, role_description)
        user_text = build_stage1_user_prompt(case_text, BASELINE_MODEL_NAME)
        response = call_llm(provider=BASELINE_PROVIDER, model_uri=BASELINE_MODEL_URI, system_text=system_text, user_text=user_text, temperature=TEMPERATURE_PROTOCOL, max_tokens=MAX_TOKENS_PROTOCOL)
        full_protocol = response['text'].strip()
        row_data['baseline_protocol'] = full_protocol
        row_data['baseline_protocol_anonymous'] = strip_author_section(full_protocol)
        row_data['baseline_model_id'] = BASELINE_MODEL_NAME
        row_data['baseline_model_uri'] = BASELINE_MODEL_URI
        row_data['baseline_provider'] = BASELINE_PROVIDER
        row_data['baseline_role_name'] = role_name
        row_data['baseline_role_description'] = role_description
        row_data['baseline_role_column'] = ROLE_COLUMN
        row_data['prompt_tokens'] = response['prompt_tokens']
        row_data['completion_tokens'] = response['completion_tokens']
        row_data['total_tokens'] = response['total_tokens']
        row_data['cached'] = response['cached']
        row_data['stop_reason'] = response.get('stop_reason', '')
        row_data['truncated'] = response.get('truncated', False)
        rows.append(row_data)
    return pd.DataFrame(rows)
result_df = run_baseline_pipeline(df)
print(result_df.shape)
result_df.head(5)


In [ ]:
def build_token_summary(result_df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame([{'model_id': BASELINE_MODEL_NAME, 'model_uri': BASELINE_MODEL_URI, 'provider': BASELINE_PROVIDER, 'role_column': ROLE_COLUMN, 'prompt_tokens': int(result_df['prompt_tokens'].fillna(0).sum()), 'completion_tokens': int(result_df['completion_tokens'].fillna(0).sum()), 'total_tokens': int(result_df['total_tokens'].fillna(0).sum()), 'n_cases': len(result_df), 'n_cached': int(result_df['cached'].fillna(False).sum())}])
token_summary_df = build_token_summary(result_df)
token_summary_df


In [ ]:
result_df_part = result_df.copy()
result_df_part['table_source'] = 'result_df'
token_summary_part = token_summary_df.copy()
token_summary_part['table_source'] = 'token_summary_df'
combined_output_df = pd.concat([result_df_part, token_summary_part], ignore_index=True, sort=False)
combined_output_df.to_csv(OUTPUT_CSV, sep=';', index=False, encoding='utf-8-sig')
print('Saved:', OUTPUT_CSV)
print('Rows (result_df):', len(result_df))
print('Total tokens:', token_summary_df['total_tokens'].iloc[0])
